# 02. Back Translation
Augment minority-class samples (HATE, OFFENSIVE) via round-trip translation: VI → EN → VI.
Only the train split is augmented — dev and test remain untouched.

## Dependencies

In [7]:
!git clone https://github.com/HoaiAn001/Vietnamese-HSD-Augmentation
%cd Vietnamese-HSD-Augmentation
!pip install -q -r requirements.txt
!pip install -q deep-translator sacrebleu

Cloning into 'Vietnamese-HSD-Augmentation'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 96 (delta 26), reused 63 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 1.91 MiB | 4.12 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/Vietnamese-HSD-Augmentation/Vietnamese-HSD-Augmentation


In [8]:
import os
import time
import pandas as pd
from tqdm.notebook import tqdm
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
from google.colab import drive, userdata
from deep_translator import GoogleTranslator
from sacrebleu.metrics import BLEU

## Setup

In [9]:
drive.mount('/content/drive')

PROJECT_DIR    = '/content/drive/MyDrive/Vietnamese_HSD_Project'
DATA_PROCESSED = f'{PROJECT_DIR}/data/processed'
DATA_AUGMENTED = f'{PROJECT_DIR}/data/augmented'
RESULTS_DIR    = f'{PROJECT_DIR}/results/figures'

os.makedirs(DATA_AUGMENTED, exist_ok=True)

token = userdata.get('HF_TOKEN')
login(token=token, add_to_git_credential=False)

REPO_ID     = 'HoaiAn001/tdtu-vietnamese-hsd'
REPO_AUG_ID = 'HoaiAn001/tdtu-hsd-aug'

TRAIN_SET = load_dataset(REPO_ID, split='tdtu_train').to_pandas()
DEV_SET   = load_dataset(REPO_ID, split='tdtu_dev').to_pandas()
TEST_SET  = load_dataset(REPO_ID, split='tdtu_test').to_pandas()

print(f'train={len(TRAIN_SET):,} | dev={len(DEV_SET):,} | test={len(TEST_SET):,}')
print(TRAIN_SET['label'].value_counts())

Mounted at /content/drive


data/tdtu_dev-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

data/tdtu_test-00000-of-00001.parquet:   0%|          | 0.00/314k [00:00<?, ?B/s]

data/tdtu_train-00000-of-00001.parquet:   0%|          | 0.00/8.60M [00:00<?, ?B/s]

Generating tdtu_dev split:   0%|          | 0/2672 [00:00<?, ? examples/s]

Generating tdtu_test split:   0%|          | 0/6680 [00:00<?, ? examples/s]

Generating tdtu_train split:   0%|          | 0/86719 [00:00<?, ? examples/s]

train=86,719 | dev=2,672 | test=6,680
label
CLEAN        61410
OFFENSIVE    19577
HATE          5732
Name: count, dtype: int64


## Back Translation

In [10]:
AUGMENT_LABELS = ['HATE', 'OFFENSIVE']
PIVOT_LANG     = 'en'
DELAY_SEC      = 0.5
BATCH_SAVE     = 100
SAMPLE_PER_LABEL = 500

def back_translate(text: str, pivot: str = 'en') -> str:
    intermediate = GoogleTranslator(source='vi', target=pivot).translate(text)
    result       = GoogleTranslator(source=pivot, target='vi').translate(intermediate)
    time.sleep(DELAY_SEC)
    return result

df_minority = (
    TRAIN_SET[TRAIN_SET['label'].isin(AUGMENT_LABELS)]
    .groupby('label', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_LABEL), random_state=42))
    .reset_index(drop=True)
)

print(f'Total samples to translate: {len(df_minority):,}')
print(df_minority['label'].value_counts())

CHECKPOINT_PATH = f'{DATA_AUGMENTED}/bt_checkpoint.csv'
augmented_rows = []
failed = []
start_index = 0

if os.path.exists(CHECKPOINT_PATH):
    print(f"\nFound checkpoint file at: {CHECKPOINT_PATH}")
    df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
    augmented_rows = df_checkpoint.to_dict('records')
    start_index = len(augmented_rows)
    print(f"Resuming translation from sample: {start_index} / {len(df_minority)}")
else:
    print(f"\nNo checkpoint found. Starting translation from scratch...")

for i in tqdm(range(start_index, len(df_minority)), desc='Back Translation'):
    row = df_minority.iloc[i]
    try:
        bt_text = back_translate(row['text'])
        if bt_text and bt_text.strip() != row['text'].strip():
            augmented_rows.append({
                'text'         : bt_text,
                'label'        : row['label'],
                'source'       : f"{row['source']}_bt_{PIVOT_LANG}",
                'original_text': row['text'],
            })
    except Exception as e:
        failed.append({'index': i, 'error': str(e), 'text': row['text']})

    if (i + 1) % BATCH_SAVE == 0:
        pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')

if len(augmented_rows) > 0:
    pd.DataFrame(augmented_rows).to_csv(CHECKPOINT_PATH, index=False, encoding='utf-8-sig')

df_bt     = pd.DataFrame(augmented_rows)
df_failed = pd.DataFrame(failed) if failed else pd.DataFrame()

print(f'\nAugmented={len(df_bt):,} | Failed={len(df_failed):,}')

/tmp/ipykernel_542/2132084388.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLE_PER_LABEL), random_state=42))


Total samples to translate: 1,000
label
HATE         500
OFFENSIVE    500
Name: count, dtype: int64

Found checkpoint file at: /content/drive/MyDrive/Vietnamese_HSD_Project/data/augmented/bt_checkpoint.csv
Resuming translation from sample: 4002 / 1000


Back Translation: 0it [00:00, ?it/s]


Augmented=4,002 | Failed=0


## Quality Check

In [12]:
samples = df_bt.sample(5, random_state=42)[['original_text', 'text', 'label']]
for _, row in samples.iterrows():
    print(f'[{row["label"]}]')
    print(f'  original : {row["original_text"]}')
    print(f'  backslation result: {row["text"]}')
    print()

[OFFENSIVE]
  original : Người ta bận họp kìa trời ơi, phải họp thì mới hết lũ dc
  backslation result: Người ta bận họp, trời ơi phải họp mới dẹp được

[HATE]
  original : Thôi ông im mẹ cái mồm đi
  backslation result: Làm ơn im mồm đi

[HATE]
  original : Thầy bói Ba cl gì gì vậy thầy, đem con susan ra thế giới à
  backslation result: Thầy bói, ông đang làm gì vậy, đưa Susan vào thế giới này à?

[HATE]
  original : Nản ba gà thật sự
  backslation result: Thực sự bực bội

[HATE]
  original : Nhìn tướng của nhà tiên tri mà tôi thấy bần tiện lưu manh thế nào í
  backslation result: Nhìn vẻ ngoài của nhà tiên tri, tôi thấy ông ta hèn hạ và thô lỗ đến thế nào



In [13]:
bleu = BLEU(effective_order=True)
refs = df_bt['original_text'].tolist()
hyps = df_bt['text'].tolist()

scores = [bleu.sentence_score(h, [r]).score for h, r in zip(hyps, refs)]
df_bt['bleu'] = scores

print(f'BLEU — mean: {df_bt["bleu"].mean():.2f} | median: {df_bt["bleu"].median():.2f}')
print(f'       min : {df_bt["bleu"].min():.2f}  | max   : {df_bt["bleu"].max():.2f}')

before = len(df_bt)
df_bt  = df_bt[df_bt['bleu'] < 100].reset_index(drop=True)
print(f'\nFiltered identical samples: {before - len(df_bt)}')
print(f'Remaining: {len(df_bt):,}')

BLEU — mean: 19.46 | median: 15.62
       min : 0.00  | max   : 100.00

Filtered identical samples: 3
Remaining: 3,999


## Save

In [14]:
df_bt.to_csv(f'{DATA_AUGMENTED}/bt_augmented.csv', index=False, encoding='utf-8-sig')
print(f'bt_augmented.csv saved: {len(df_bt):,} samples')

if len(df_failed) > 0:
    df_failed.to_csv(f'{DATA_AUGMENTED}/bt_failed.csv', index=False, encoding='utf-8-sig')
    print(f'bt_failed.csv saved: {len(df_failed):,} samples')

keep_cols   = ['text', 'label', 'source', 'split']
TRAIN_BT    = pd.concat([TRAIN_SET, df_bt[keep_cols]], ignore_index=True)
TRAIN_BT    = TRAIN_BT.sample(frac=1, random_state=42).reset_index(drop=True)

TRAIN_BT.to_csv(f'{DATA_AUGMENTED}/train_bt.csv', index=False, encoding='utf-8-sig')

print(f'\ntrain_bt.csv saved: {len(TRAIN_BT):,} samples')
print(TRAIN_BT['label'].value_counts())

bt_augmented.csv saved: 3,999 samples

train_bt.csv saved: 90,718 samples
label
CLEAN        61410
OFFENSIVE    21092
HATE          8216
Name: count, dtype: int64


## Push to HuggingFace Hub

In [15]:
hf_token = userdata.get('HF_TOKEN')

HF_USERNAME = 'HoaiAn001'
REPO_ID     = f'{HF_USERNAME}/tdtu-vietnamese-hsd'

hf_bt = DatasetDict({
    'train'     : Dataset.from_pandas(TRAIN_BT.reset_index(drop=True)),
    'validation': Dataset.from_pandas(DEV_SET.reset_index(drop=True)),
    'test'      : Dataset.from_pandas(TEST_SET.reset_index(drop=True)),
})

hf_bt.push_to_hub(REPO_ID, config_name='bt', private=False, token=hf_token)
print(f'https://huggingface.co/datasets/{REPO_ID}')

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/91 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   6%|5         |  526kB / 9.12MB            

...

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  13%|#3        | 41.7kB /  318kB            

https://huggingface.co/datasets/HoaiAn001/tdtu-vietnamese-hsd


## Summary

In [19]:
orig_counts = TRAIN_SET['label'].value_counts()
bt_counts   = TRAIN_BT['label'].value_counts()

print('Label distribution before vs after BT:')
print(f'{"label":<12} {"before":>8} {"after":>8} {"added":>8}')
print('-' * 40)
for label in ['CLEAN', 'OFFENSIVE', 'HATE']:
    before = orig_counts.get(label, 0)
    after  = bt_counts.get(label, 0)
    print(f'{label:<12} {before:>8,} {after:>8,} {after-before:>+8,}')

print(f'\nBLEU (mean): {df_bt["bleu"].mean():.2f}')
print(f'Failed     : {len(df_failed):,}')

Label distribution before vs after BT:
label          before    after    added
----------------------------------------
CLEAN          61,410   61,410       +0
OFFENSIVE      19,577   21,092   +1,515
HATE            5,732    8,216   +2,484

BLEU (mean): 19.40
Failed     : 0
